# Imports

In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from torch.utils.data import Dataset
from torch.optim import AdamW
from transformers import DataCollatorForLanguageModeling
import matplotlib as plt
from transformers import BertTokenizer, BertModel
import torch
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import re

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

Mounted at /content/drive


## Data preparation

In [3]:
df_ref = pd.read_csv("files/df_ml_good_with_features.csv")

df_ref_amy = df_ref[df_ref['class'] == 1]
df_ref_nonamy = df_ref[df_ref['class'] == 0]

df_protgpt2 = pd.read_csv("./files/final_datasets/protgpt_final.csv")
df_cvae = pd.read_csv("./files/final_datasets/cvae_final.csv")

In [ ]:
selected_features = ['beta_propensity', 'proline_fraction', 'AAT', 'net_charge', 'TA', 'polar_fraction', 'a3vSA']

X_real = df_ref[selected_features]
y_real = df_ref["class"]

X_train_ref, X_test, y_train_ref, y_test = train_test_split(
    X_real, y_real,
    test_size=0.2,
    random_state=42,
    stratify=y_real
)

In [ ]:
df_protgpt2['class'] = 1
df_cvae['class'] = 1

train_datasets = {
    "Real Only": (X_train_ref, y_train_ref),
    "Real + ProtGPT2": (
        pd.concat([X_train_ref, df_protgpt2[selected_features]]),
        pd.concat([y_train_ref, df_protgpt2["class"]])
    ),
    "Real + cVAE": (
        pd.concat([X_train_ref, df_cvae[selected_features]]),
        pd.concat([y_train_ref, df_cvae["class"]])
    )
}

results = []

for name, (X_train_curr, y_train_curr) in train_datasets.items():
    print(f"\n--- Trening na zbiorze: {name} ---")

    # Skalowanie
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_curr)
    X_test_scaled = scaler.transform(X_test)

    # Model (użyjmy Random Forest jako przykładu, możesz tu wstawić funkcję trenującą MLP/CNN)
    model = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
    model.fit(X_train_scaled, y_train_curr)

    # Predykcja na STAŁYM zbiorze testowym
    probs = model.predict_proba(X_test_scaled)[:, 1]
    preds = model.predict(X_test_scaled)

    # Metryki
    auc = roc_auc_score(y_test, probs)
    f1 = f1_score(y_test, preds)
    acc = accuracy_score(y_test, preds)

    results.append({
        "Dataset": name,
        "AUC": auc,
        "F1": f1,
        "Accuracy": acc
    })

    print(f"Wyniki dla {name}: AUC={auc:.4f}, F1={f1:.4f}")

# Wyświetlenie zbiorczego raportu
df_results = pd.DataFrame(results)
print("\nPORÓWNANIE KOŃCOWE:")
print(df_results.sort_values(by="F1", ascending=False))

## Import of amyloid set

In [ ]:
path = "/content/drive/MyDrive/df_ml_good_with_features.csv"
df = pd.read_csv(path)

print(df.shape)
df.head()

(1934, 21)


,id,sequence,length,class,merge_id,a3vSA,AAT,Na4vSS,THSA,nHS,...,net_charge,hydrophobicity,polar_fraction,nonpolar_fraction,acidic_fraction,basic_fraction,aromatic_fraction,beta_propensity,seq_entropy,proline_fraction
0,GI_171848907_PDB_2RNM_A__1_1,GRNSAKDIRTEERARVQLGNV,21,1,GI_171848907_PDB_2RNM_A__1_1_0,-0.457,0.190,-0.494,0.190,1.0,...,2.0,-1.185714,0.238095,0.380952,0.142857,0.238095,0.000000,0.953333,3.535175,0.000000
1,GI_342871650_GB_EGU74155_1_1,VRIYAKDIKSEEMARVRVGNE,21,1,GI_342871650_GB_EGU74155_1_1_0,-0.160,2.140,-0.165,1.871,1.0,...,1.0,-0.676190,0.142857,0.428571,0.190476,0.238095,0.047619,0.990000,3.427333,0.000000
2,GI_342887385_GB_EGU86897_1_1,GKNSAGRINGPGMVNIGNS,19,1,GI_342887385_GB_EGU86897_1_1_0,-0.256,1.438,-0.256,1.438,1.0,...,2.0,-0.563158,0.315789,0.578947,0.000000,0.105263,0.000000,0.937368,3.005315,0.052632
3,GI_347837243_EMB_CCD5181_1_1,HRIKIGKVTQASNAKAVIGVH,21,1,GI_347837243_EMB_CCD5181_1_1_0,-0.001,4.054,-0.008,4.054,2.0,...,4.2,-0.019048,0.190476,0.523810,0.000000,0.285714,0.000000,1.081429,3.296148,0.000000
4,GI_475677570_GB_EMT74561_1_1,VRNYASEIKGDEDAKVRLGND,21,1,GI_475677570_GB_EMT74561_1_1_0,-0.436,0.570,-0.435,0.000,0.0,...,-1.0,-1.138095,0.190476,0.380952,0.238095,0.190476,0.047619,0.912381,3.499228,0.000000


In [ ]:
# Most important features (from SHAP) - 0.7% of total impact on model
selected_features = ['beta_propensity', 'proline_fraction', 'AAT', 'net_charge', 'TA', 'polar_fraction', 'a3vSA']

# Models trained on base set

## Baseline


In [ ]:
X = df[selected_features].copy()
y = df["class"]

print("X shape:", X.shape)

X shape: (1934, 7)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_scaled, y_train)

rf_probs = rf.predict_proba(X_test_scaled)[:, 1]
rf_preds = rf.predict(X_test_scaled)

print("RF ROC AUC:", roc_auc_score(y_test, rf_probs))
print("RF F1:", f1_score(y_test, rf_preds))
print("RF Accuracy:", accuracy_score(y_test, rf_preds))

print(classification_report(y_test, rf_preds, digits=2))

RF ROC AUC: 0.7860161557802385
RF F1: 0.7481662591687042
RF Accuracy: 0.7338501291989664
              precision    recall  f1-score   support

           0       0.73      0.70      0.72       186
           1       0.74      0.76      0.75       201

    accuracy                           0.73       387
   macro avg       0.73      0.73      0.73       387
weighted avg       0.73      0.73      0.73       387



### Logistic Regression

In [ ]:
lr = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

lr.fit(X_train_scaled, y_train)

lr_probs = lr.predict_proba(X_test_scaled)[:, 1]
lr_preds = lr.predict(X_test_scaled)

print("LR ROC AUC:", roc_auc_score(y_test, lr_preds))
print("LR F1:", f1_score(y_test, lr_preds))
print("LR Accuracy:", accuracy_score(y_test, lr_preds))

print(classification_report(y_test, lr_preds, digits=2))

LR ROC AUC: 0.6807896003851708
LR F1: 0.7007299270072993
LR Accuracy: 0.6821705426356589
              precision    recall  f1-score   support

           0       0.68      0.65      0.66       186
           1       0.69      0.72      0.70       201

    accuracy                           0.68       387
   macro avg       0.68      0.68      0.68       387
weighted avg       0.68      0.68      0.68       387



### XGBoost

In [ ]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

xgb.fit(X_train_scaled, y_train)
xgb_probs = xgb.predict_proba(X_test_scaled)[:, 1]
xgb_preds = xgb.predict(X_test_scaled)

print("XGB ROC AUC:", roc_auc_score(y_test, xgb_preds))
print("XGB F1:", f1_score(y_test, xgb_preds))
print("XGB Accuracy:", accuracy_score(y_test, xgb_preds))

print(classification_report(y_test, xgb_preds, digits=2))

XGB ROC AUC: 0.7191060824907719
XGB F1: 0.7403846153846154
XGB Accuracy: 0.7209302325581395
              precision    recall  f1-score   support

           0       0.73      0.67      0.70       186
           1       0.72      0.77      0.74       201

    accuracy                           0.72       387
   macro avg       0.72      0.72      0.72       387
weighted avg       0.72      0.72      0.72       387



## Deep models

#### MLP

In [ ]:
#2-warstwowy MLP (2 hidden layers)
#z dropoutem i early stoppingiem

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_scaled, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

X_tr_t = torch.tensor(X_tr, dtype=torch.float32)
y_tr_t = torch.tensor(y_tr.values, dtype=torch.float32).view(-1, 1)

X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val.values, dtype=torch.float32).view(-1, 1)

X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_t = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=32, shuffle=True)

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(7, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

model = MLP()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

best_val_loss = float("inf")
patience = 15
counter = 0
best_model_state = None

for epoch in range(300):
    model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_logits = model(X_val_t)
        val_loss = criterion(val_logits, y_val_t)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = model.state_dict()
        counter = 0
    else:
        counter += 1

    if counter >= patience:
        print(f"Early stopping at epoch {epoch}")
        break

model.load_state_dict(best_model_state)

model.eval()
with torch.no_grad():
    test_logits = model(X_test_t)
    probs = torch.sigmoid(test_logits).numpy().flatten()

thresholds = np.linspace(0.1, 0.9, 100)
f1_scores = []

for t in thresholds:
    temp_preds = (probs > t).astype(int)
    f1_scores.append(f1_score(y_test, temp_preds))

best_t = thresholds[np.argmax(f1_scores)]
print("Best threshold:", best_t)
print("Best F1:", max(f1_scores))

preds = (probs > best_t).astype(int)

print("ROC AUC:", roc_auc_score(y_test, probs))
print(classification_report(y_test, preds, digits=4))

Early stopping at epoch 65
Best threshold: 0.5686868686868687
Best F1: 0.7597535934291582
ROC AUC: 0.7757716792382175
              precision    recall  f1-score   support

           0     0.8416    0.4570    0.5923       186
           1     0.6469    0.9204    0.7598       201

    accuracy                         0.6977       387
   macro avg     0.7442    0.6887    0.6760       387
weighted avg     0.7404    0.6977    0.6793       387



#### CNN (sekwencja) + MLP (cechy)
SEQ: Embedding -> Conv1D -> ReLU -> MaxPool -> Conv1D -> GlobalMaxPool

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

max_seq_len = 50
amino_acids = 'ACDEFGHIKLMNPQRSTVWY'

aa_to_idx = {aa:i+1 for i, aa in enumerate(amino_acids)}

class HybridDataset(Dataset):
    def __init__(self, seqs, feats, labels):
        self.seqs = seqs
        self.feats = feats
        self.labels = labels

    def encode_seq(self, seq):
        seq = seq[:max_seq_len]
        encoded = [aa_to_idx.get(aa, 0) for aa in seq]
        if len(encoded) < max_seq_len:
            encoded += [0]*(max_seq_len - len(encoded))
        return torch.tensor(encoded, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            self.encode_seq(self.seqs[idx]),
            torch.tensor(self.feats[idx], dtype=torch.float32),
            torch.tensor(self.labels[idx], dtype=torch.float32)
        )

seqs = df["sequence"].values
features = df[selected_features].values
labels = df["class"].values

scaler = StandardScaler()
features = scaler.fit_transform(features)

X_seq_tr, X_seq_test, X_feat_tr, X_feat_test, y_tr, y_test = train_test_split(
    seqs, features, labels, test_size=0.2,
    random_state=42, stratify=labels
)

X_seq_tr, X_seq_val, X_feat_tr, X_feat_val, y_tr, y_val = train_test_split(
    X_seq_tr, X_feat_tr, y_tr,
    test_size=0.2, random_state=42, stratify=y_tr
)

train_loader = DataLoader(
    HybridDataset(X_seq_tr, X_feat_tr, y_tr),
    batch_size=32, shuffle=True
)

val_loader = DataLoader(
    HybridDataset(X_seq_val, X_feat_val, y_val),
    batch_size=64
)

test_loader = DataLoader(
    HybridDataset(X_seq_test, X_feat_test, y_test),
    batch_size=64
)

In [ ]:
class CNNHybrid(nn.Module):
    def __init__(self):
        super().__init__()

        # embedding aminokwasów
        self.embedding = nn.Embedding(len(amino_acids)+1, 32, padding_idx=0)

        # CNN
        self.conv = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveMaxPool1d(1)
        )

        # cechy
        self.feat_net = nn.Sequential(
            nn.Linear(len(selected_features), 32),
            nn.ReLU()
        )

        # klasyfikator
        self.classifier = nn.Sequential(
            nn.Linear(128 + 32, 64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, 1)
        )

    def forward(self, seq, feats):
        x = self.embedding(seq)          # (batch, L, 32)
        x = x.permute(0,2,1)             # (batch, 32, L)
        x = self.conv(x).squeeze(-1)    # (batch, 128)

        f = self.feat_net(feats)

        combined = torch.cat([x, f], dim=1)
        return self.classifier(combined)

In [ ]:
model = CNNHybrid().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

best_val = float("inf")
patience = 15
counter = 0

for epoch in range(200):
    model.train()
    for seq, feat, y in train_loader:
        seq, feat, y = seq.to(device), feat.to(device), y.to(device).unsqueeze(1)

        optimizer.zero_grad()
        logits = model(seq, feat)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for seq, feat, y in val_loader:
            seq, feat, y = seq.to(device), feat.to(device), y.to(device).unsqueeze(1)
            logits = model(seq, feat)
            val_loss += criterion(logits, y).item()

    if val_loss < best_val:
        best_val = val_loss
        best_state = model.state_dict()
        counter = 0
    else:
        counter += 1

    if counter >= patience:
        print("Early stopping")
        break

model.load_state_dict(best_state)

model.eval()
probs = []
true = []

with torch.no_grad():
    for seq, feat, y in test_loader:
        seq, feat = seq.to(device), feat.to(device)
        logits = model(seq, feat)
        p = torch.sigmoid(logits).cpu().numpy().flatten()
        probs.extend(p)
        true.extend(y.numpy())

probs = np.array(probs)
true = np.array(true)

preds = (probs > 0.5).astype(int)

print("ROC AUC:", roc_auc_score(true, probs))
print(classification_report(true, (probs > 0.5).astype(int), digits=4))

Early stopping
ROC AUC: 0.8752741668004066
              precision    recall  f1-score   support

         0.0     0.7877    0.7581    0.7726       186
         1.0     0.7837    0.8109    0.7971       201

    accuracy                         0.7855       387
   macro avg     0.7857    0.7845    0.7848       387
weighted avg     0.7856    0.7855    0.7853       387



#### Model oparty na embeddingach białkowych (ProtBERT)

sekwencja -> ProtBERT -> embedding (1024) -> MLP -> klasyfikacja

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = BertTokenizer.from_pretrained("Rostlab/prot_bert", do_lower_case=False)
model_bert = BertModel.from_pretrained("Rostlab/prot_bert")
model_bert = model_bert.to(device)
model_bert.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/81.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/361 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/487 [00:00<?, ?it/s]

BertModel LOAD REPORT from: Rostlab/prot_bert
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30, 1024, padding_idx=0)
    (position_embeddings): Embedding(40000, 1024)
    (token_type_embeddings): Embedding(2, 1024)
    (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-29): 30 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=1024, out_features=1024, bias=True)
            (key): Linear(in_features=1024, out_features=1024, bias=True)
            (value): Linear(in_features=1024, out_features=1024, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=1024, out_features=1024, bias=True)
            (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.0, inpla

In [ ]:
def preprocess_sequence(seq):
    seq = seq.upper()
    seq = re.sub(r"[UZOB]", "X", seq)  # rzadkie aminokwasy
    return " ".join(list(seq))

In [ ]:
def get_embeddings(sequences, batch_size=16):
    embeddings = []

    for i in tqdm(range(0, len(sequences), batch_size)):
        batch_seqs = sequences[i:i+batch_size]
        batch_seqs = [preprocess_sequence(seq) for seq in batch_seqs]

        encoded = tokenizer(batch_seqs,
                            return_tensors="pt",
                            padding=True,
                            truncation=True,
                            max_length=512)

        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)

        with torch.no_grad():
            outputs = model_bert(input_ids=input_ids,
                                 attention_mask=attention_mask)

        # (batch, seq_len, hidden_dim)
        hidden_states = outputs.last_hidden_state

        # mean pooling (ignorujemy padding)
        mask = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
        summed = torch.sum(hidden_states * mask, dim=1)
        counts = torch.clamp(mask.sum(dim=1), min=1e-9)
        mean_pooled = summed / counts

        embeddings.append(mean_pooled.cpu().numpy())

    return np.vstack(embeddings)

In [ ]:
sequences = df["sequence"].tolist()
X_embed = get_embeddings(sequences)
X_final = np.concatenate([X_embed, df[selected_features].values], axis=1)

y = df["class"].values

print("Embedding shape:", X_final.shape)

100%|██████████| 121/121 [12:28<00:00,  6.18s/it]

Embedding shape: (1934, 1031)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42
)

train_ds = TensorDataset(
    torch.tensor(X_tr, dtype=torch.float32),
    torch.tensor(y_tr, dtype=torch.float32).view(-1,1)
)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32).view(-1,1)

X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).view(-1,1)

In [ ]:
class ProtBERT_MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
model = ProtBERT_MLP(X_train.shape[1]).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

best_val = float("inf")
patience = 10
counter = 0

for epoch in range(100):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_logits = model(X_val_t.to(device))
        val_loss = criterion(val_logits, y_val_t.to(device))

    if val_loss < best_val:
        best_val = val_loss
        best_state = model.state_dict()
        counter = 0
    else:
        counter += 1

    if counter >= patience:
        print("Early stopping")
        break

model.load_state_dict(best_state)

Early stopping


<All keys matched successfully>

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(X_test_t.to(device))
    probs = torch.sigmoid(logits).cpu().numpy().flatten()

preds = (probs > 0.5).astype(int)

print("ROC AUC:", roc_auc_score(y_test, probs))
print(classification_report(y_test, preds, digits=4))

ROC AUC: 0.8693628631038356
              precision    recall  f1-score   support

           0     0.7766    0.7849    0.7807       186
           1     0.7990    0.7910    0.7950       201

    accuracy                         0.7881       387
   macro avg     0.7878    0.7880    0.7879       387
weighted avg     0.7882    0.7881    0.7882       387



## Model z poprzedniej pracy - inne zadanie, nie klasyfikacja binarna -> odmaskowywanie tokenów

In [ ]:
def format_sequence(seq):
    """
    Zamienia sekwencję aminokwasów na format z rozdzielonymi spacjami,
    wymagany przez ProtBERT.
    """
    seq = seq.upper()           # duże litery
    seq = " ".join(list(seq))   # wstaw spacje między aminokwasy
    return seq

tokenizer = BertTokenizer.from_pretrained(
    "Rostlab/prot_bert",
    do_lower_case=False
)

def mask_tokens(inputs, tokenizer, mask_prob=0.2):
    """
    Maskowanie tokenów dla zadania rekonstrukcji sekwencji
    """
    labels = inputs.clone()
    special_tokens = {0, 2, 3}  # CLS, PAD, SEP

    mask = torch.bernoulli(torch.full(labels.shape, mask_prob)).bool()
    special_mask = torch.zeros_like(mask, dtype=torch.bool)
    for token_id in special_tokens:
        special_mask |= (inputs == token_id)

    mask[special_mask] = False
    inputs[mask] = tokenizer.mask_token_id
    labels[~mask] = -100
    return inputs, labels

# --- Dataset z sekwencjami i dodatkowymi cechami ---
class ProteinDataset(Dataset):
    def __init__(self, df, selected_features):
        self.sequences = df['sequence'].tolist()
        self.features = df[selected_features].values.astype(float)
        self.labels = df['sequence'].tolist()  # w rekonstrukcji labels = sekwencja
        self.tokenizer = tokenizer
        self.max_len = max_seq_len

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = format_sequence(self.sequences[idx])
        encoded = self.tokenizer(
            seq,
            padding="max_length",
            truncation=True,
            max_length=self.max_seq_len,
            return_tensors="pt"
        )
        input_ids = encoded['input_ids'].squeeze(0)
        attention_mask = encoded['attention_mask'].squeeze(0)

        input_ids_masked, labels = mask_tokens(input_ids, self.tokenizer, mask_prob)

        features = torch.tensor(self.features[idx], dtype=torch.float32)
        labels = labels.long()  # CrossEntropyLoss wymaga Long

        return {
            'input_ids': input_ids_masked,
            'attention_mask': attention_mask,
            'additional_features': features,
            'labels': labels
        }

dataset = ProteinDataset(df, selected_features)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/81.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
class CustomCollator:
    def __init__(self, tokenizer):
        self.mlm_collator = DataCollatorForLanguageModeling(
            tokenizer=tokenizer,
            mlm=True,
            mlm_probability=0.15
        )

    def __call__(self, batch):
        additional_features = torch.stack([item["additional_features"] for item in batch])

        bert_batch = self.mlm_collator([
            {
                "input_ids": item["input_ids"],
                "attention_mask": item["attention_mask"]
            }
            for item in batch
        ])

        bert_batch["additional_features"] = additional_features
        return bert_batch

In [ ]:
class ProteinDataset(Dataset):
    def __init__(self, sequences, additional_features, tokenizer, max_len=512, mask_prob=0.15):
        self.sequences = sequences
        self.features = additional_features
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.mask_prob = mask_prob

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        feat = self.features[idx]

        # przygotowanie sekwencji do ProtBERT
        seq_formatted = " ".join(list(seq.upper()))
        encoded = tokenizer(
            seq_formatted,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        input_ids = encoded["input_ids"].squeeze(0)
        attention_mask = encoded["attention_mask"].squeeze(0)

        # maskowanie tokenów
        input_ids_masked, labels = mask_tokens(input_ids.clone(), tokenizer, mask_prob=self.mask_prob)

        return {
            "input_ids": input_ids_masked,
            "attention_mask": attention_mask,
            "labels": labels,
            "additional_features": torch.tensor(feat, dtype=torch.float32)
        }

In [ ]:
class ProteinTransformerForReconstruction2(nn.Module):
    def __init__(self, additional_feature_dim):
        super(ProteinTransformerForReconstruction2, self).__init__()
        # Pre-trained BERT model for proteins
        self.bert = BertModel.from_pretrained('Rostlab/prot_bert')

        self.fc_bert = nn.Linear(self.bert.config.hidden_size, 256)
        self.fc_additional = nn.Linear(additional_feature_dim, 256)

        # Attention mechanism for additional features
        self.additional_attention = nn.MultiheadAttention(embed_dim=256, num_heads=4, batch_first=True)

        # Final fully connected layer to reconstruct the sequence
        self.fc_output = nn.Linear(256 + 256, self.bert.config.vocab_size)

    def forward(self, input_ids, attention_mask, additional_features):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state  # Shape: (batch_size, sequence_length, hidden_size)

        bert_representation = self.fc_bert(sequence_output)  # Shape: (batch_size, sequence_length, 256)
        additional_representation = self.fc_additional(additional_features)  # Shape: (batch_size, 256)

        # Expand the additional_representation to match the sequence length
        # unsqueeze(1) adds a sequence dimension; repeat copies it across the sequence length
        additional_representation_expanded = additional_representation.unsqueeze(1).repeat(1, sequence_output.size(1), 1)

        # Apply attention to the additional features
        # Query is the BERT representation; key and value are derived from the additional representation
        attn_output, _ = self.additional_attention(
            query=bert_representation,
            key=additional_representation_expanded,
            value=additional_representation_expanded
        )

        # Combine the attention output with the BERT representation
        combined_representation = torch.cat((attn_output, bert_representation), dim=-1)  # Shape: (batch_size, sequence_length, 512)

        logits = self.fc_output(combined_representation)  # Shape: (batch_size, sequence_length, vocab_size)

        return logits

In [ ]:
dataset_train = ProteinDataset(
    sequences=df['sequence'].tolist(),
    additional_features=df[selected_features].values,
    tokenizer=tokenizer,
    max_len=512,
    mask_prob=0.15
)

In [ ]:
collator = CustomCollator(tokenizer)

dataloader_train = DataLoader(
    dataset_train,
    batch_size=8,
    shuffle=True,
    collate_fn=collator
)

combined_features = df[selected_features].values.astype(np.float32)

In [ ]:
model = ProteinTransformerForReconstruction2(combined_features.shape[1])
optimizer = AdamW(model.parameters(), lr=1e-5)
criterion = nn.CrossEntropyLoss(ignore_index=-100)

patience = 2
best_val_loss = float('inf')
epochs_no_improve = 0

num_epochs = 30

avg_loss_values = []
val_loss_values = []
sample_counts = []

model.train()
total_samples = 0

for epoch in range(num_epochs):
    total_loss = 0
    for batch in dataloader_train:
        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']
        labels = batch['labels']
        additional_features = batch['additional_features'].float()

        logits = model(input_ids=input_ids, attention_mask=attention_mask, additional_features=additional_features)
        loss = criterion(logits.view(-1, logits.size(-1)), labels.view(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_samples += input_ids.size(0)

    avg_loss = total_loss / len(dataloader_train)
    avg_loss_values.append(avg_loss)
    sample_counts.append(total_samples)

    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for val_batch in dataloader_val:
            val_input_ids = val_batch['input_ids']
            val_attention_mask = val_batch['attention_mask']
            val_labels = val_batch['labels']
            val_additional_features = val_batch['additional_features'].float()


            val_outputs = model(input_ids=val_input_ids, attention_mask=val_attention_mask, additional_features=val_additional_features)
            val_loss = loss = criterion(val_outputs.view(-1, val_outputs.size(-1)), val_labels.view(-1))
            total_val_loss += val_loss.item()

    avg_val_loss = total_val_loss / len(dataloader_val)
    val_loss_values.append(avg_val_loss)

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), 'ProteinTransformerForReconstruction_additional_heads_training_crossEntropy2.pth')
    else:
        epochs_no_improve += 1

    print(f'Epoch {epoch + 1}/{num_epochs}, Samples Processed: {total_samples}, '
          f'Avg Loss: {avg_loss:.4f}, Val Loss: {avg_val_loss:.4f}')

    if epochs_no_improve >= patience:
        print(f"Early stopping triggered at epoch {epoch + 1}")
        break

    model.train()

torch.save(model.state_dict(), 'ProteinTransformerForReconstruction_additional_heads_training_crossEntropy.pth')

config.json:   0%|          | 0.00/361 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/487 [00:00<?, ?it/s]

BertModel LOAD REPORT from: Rostlab/prot_bert
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(sample_counts, avg_loss_values, marker='o', linestyle='-', color='b', label='Train Loss')
plt.plot(sample_counts, val_loss_values, marker='o', linestyle='--', color='r', label='Validation Loss')
plt.title('Krzywa Uczenia - Dotrenowanie modelu ProteinTransformerForReconstruction\n z funkcją straty CrossEntropy')
plt.xlabel('Liczba przetworzonych próbek')
plt.ylabel('Średnia strata')
plt.legend()
plt.grid(True)
plt.show()
plt.savefig('learning_curve_model_ProteinTransformerForReconstruction_additional_heads_training_crossEntropy2.svg', format='svg')